# Laboratorio 3: regresión con Bike Sharing — PyTorch

**Autor:** Nataniel Mauricio Arapa Estrada  
**Materia:** SIS420 — Inteligencia Artificial I

Se predice la cantidad horaria de alquileres (`cnt`) con tres variantes implementadas en PyTorch:

1. Regresión lineal multivariable (`nn.Linear`).
2. Regresión polinómica de grado 2 (`nn.Linear` sobre $X$ y $X^2$).
3. Solución analítica por mínimos cuadrados (`torch.linalg.lstsq`).

Pandas se utiliza para leer el CSV y Matplotlib para las gráficas. La división, transformaciones numéricas, entrenamiento, predicción y métricas se hacen con tensores de PyTorch.


## 1. Configuración y carga de datos


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def buscar_archivo(nombre, carpeta_lab):
    bases = [
        Path.cwd(),
        Path.cwd() / carpeta_lab,
        Path.cwd().parent / carpeta_lab,
        Path("/content/drive/MyDrive/Colab Notebooks/machine learning/datasets"),
        Path("/content/gdrive/MyDrive/Colab Notebooks/machine learning/datasets"),
    ]
    for base in bases:
        candidato = base / nombre
        if candidato.exists():
            return candidato
    raise FileNotFoundError(f"No se encontró {nombre}. Colócalo junto al notebook o en Drive.")


ruta_hour = buscar_archivo("hour.csv", "Lab3")
datos = pd.read_csv(ruta_hour)
print(f"PyTorch {torch.__version__} | Dispositivo: {DEVICE}")
print(f"Dataset: {ruta_hour}")
print(f"Dimensiones: {datos.shape}")
display(datos.head())


## 2. Selección y división

No se usan `casual` ni `registered`, ya que su suma forma exactamente `cnt` y produciría fuga directa de la respuesta. Se conservan las mismas 12 características del cuadernillo original.


In [ ]:
nombres = [
    "season", "yr", "mnth", "hr", "holiday", "weekday",
    "workingday", "weathersit", "temp", "atemp", "hum", "windspeed",
]
X = torch.tensor(datos[nombres].to_numpy(dtype="float32"), dtype=DTYPE)
y = torch.tensor(datos["cnt"].to_numpy(dtype="float32"), dtype=DTYPE)


def dividir(X, y, proporcion_prueba=0.20, semilla=SEED):
    generador = torch.Generator().manual_seed(semilla)
    indices = torch.randperm(len(y), generator=generador)
    corte = int(len(y) * (1 - proporcion_prueba))
    train_idx, test_idx = indices[:corte], indices[corte:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X_train, X_test, y_train, y_test = dividir(X, y)

x_media = X_train.mean(dim=0, keepdim=True)
x_std = X_train.std(dim=0, unbiased=False, keepdim=True).clamp_min(1e-8)
X_train_n = (X_train - x_media) / x_std
X_test_n = (X_test - x_media) / x_std

y_media = y_train.mean()
y_std = y_train.std(unbiased=False).clamp_min(1e-8)
y_train_n = (y_train - y_media) / y_std

print(f"Entrenamiento: {X_train.shape} | Prueba: {X_test.shape}")
print("Las estadísticas de normalización se calcularon únicamente con entrenamiento.")


## 3. Funciones PyTorch de entrenamiento y evaluación


In [ ]:
def entrenar(modelo, X_train, y_train, epocas=150, tasa=0.01, lote=512, semilla=SEED):
    generador = torch.Generator().manual_seed(semilla)
    cargador = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=lote,
        shuffle=True,
        generator=generador,
    )
    criterio = nn.MSELoss()
    optimizador = torch.optim.Adam(modelo.parameters(), lr=tasa)
    historial = []

    for _ in range(epocas):
        modelo.train()
        perdida_acumulada = 0.0
        for xb, yb in cargador:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizador.zero_grad()
            pred = modelo(xb).squeeze(1)
            perdida = criterio(pred, yb)
            perdida.backward()
            optimizador.step()
            perdida_acumulada += perdida.item() * len(yb)
        historial.append(perdida_acumulada / len(X_train))
    return historial


def predecir_desnormalizado(modelo, X_normalizado):
    modelo.eval()
    with torch.inference_mode():
        pred_n = modelo(X_normalizado.to(DEVICE)).cpu().squeeze(1)
    return pred_n * y_std + y_media


def metricas(y_real, y_pred):
    error = y_pred.flatten() - y_real.flatten()
    mse = error.square().mean()
    mae = error.abs().mean()
    rmse = mse.sqrt()
    r2 = 1 - error.square().sum() / (
        (y_real - y_real.mean()).square().sum().clamp_min(1e-12)
    )
    return {
        "MAE": mae.item(), "MSE": mse.item(),
        "RMSE": rmse.item(), "R2": r2.item(),
    }


## 4. Regresión lineal multivariable


In [ ]:
torch.manual_seed(SEED)
modelo_lineal = nn.Linear(X_train_n.shape[1], 1).to(DEVICE)
historial_lineal = entrenar(
    modelo_lineal, X_train_n, y_train_n,
    epocas=150, tasa=0.01, lote=512,
)
pred_lineal = predecir_desnormalizado(modelo_lineal, X_test_n)
resultados_lineal = metricas(y_test, pred_lineal)

print(f"Pérdida normalizada final: {historial_lineal[-1]:.6f}")
print(resultados_lineal)


## 5. Regresión polinómica de grado 2

Se amplía cada fila de entrada como $[x_1,\ldots,x_n,x_1^2,\ldots,x_n^2]$. Se normalizan estas nuevas columnas usando solamente el conjunto de entrenamiento.


In [ ]:
X_train_poly = torch.cat([X_train, X_train.square()], dim=1)
X_test_poly = torch.cat([X_test, X_test.square()], dim=1)
nombres_poly = nombres + [f"{nombre}²" for nombre in nombres]

poly_media = X_train_poly.mean(dim=0, keepdim=True)
poly_std = X_train_poly.std(dim=0, unbiased=False, keepdim=True).clamp_min(1e-8)
X_train_poly_n = (X_train_poly - poly_media) / poly_std
X_test_poly_n = (X_test_poly - poly_media) / poly_std

torch.manual_seed(SEED)
modelo_poly = nn.Linear(X_train_poly_n.shape[1], 1).to(DEVICE)
historial_poly = entrenar(
    modelo_poly, X_train_poly_n, y_train_n,
    epocas=180, tasa=0.01, lote=512,
)
pred_poly = predecir_desnormalizado(modelo_poly, X_test_poly_n)
resultados_poly = metricas(y_test, pred_poly)

print(f"Pérdida normalizada final: {historial_poly[-1]:.6f}")
print(resultados_poly)


## 6. Solución analítica con PyTorch

`torch.linalg.lstsq` resuelve $\min_\theta\|X\theta-y\|_2$ sin invertir explícitamente $X^TX$, lo que evita el problema de singularidad que podía provocar `np.linalg.inv` en el cuadernillo anterior.


In [ ]:
X_train_aug = torch.cat(
    [torch.ones((len(X_train_n), 1), dtype=torch.float64), X_train_n.double()],
    dim=1,
)
X_test_aug = torch.cat(
    [torch.ones((len(X_test_n), 1), dtype=torch.float64), X_test_n.double()],
    dim=1,
)
theta = torch.linalg.lstsq(
    X_train_aug,
    y_train_n.double().unsqueeze(1),
    driver="gelsd",
).solution
pred_lstsq_n = (X_test_aug @ theta).squeeze(1).float()
pred_lstsq = pred_lstsq_n * y_std + y_media
resultados_lstsq = metricas(y_test, pred_lstsq)
print(resultados_lstsq)


## 7. Comparación y visualización


In [ ]:
tabla_resultados = pd.DataFrame([
    {"Modelo": "Lineal — nn.Linear", **resultados_lineal},
    {"Modelo": "Polinómico grado 2 — nn.Linear", **resultados_poly},
    {"Modelo": "Lineal — torch.linalg.lstsq", **resultados_lstsq},
])
display(tabla_resultados.round(4))

comparacion = pd.DataFrame({
    "Real": y_test[:20].tolist(),
    "Lineal": pred_lineal[:20].tolist(),
    "Polinómica": pred_poly[:20].tolist(),
    "lstsq": pred_lstsq[:20].tolist(),
})
display(comparacion.round(2))


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
ejes[0].plot(historial_lineal, label="Lineal")
ejes[0].plot(historial_poly, label="Polinómica")
ejes[0].set(
    title="Curvas de entrenamiento", xlabel="Época", ylabel="MSE normalizado"
)
ejes[0].legend()
ejes[0].grid(alpha=0.3)

ejes[1].scatter(y_test.tolist(), pred_poly.tolist(), s=12, alpha=0.25)
limite_min = min(y_test.min().item(), pred_poly.min().item())
limite_max = max(y_test.max().item(), pred_poly.max().item())
ejes[1].plot(
    [limite_min, limite_max], [limite_min, limite_max],
    color="crimson", linestyle="--", label="Predicción perfecta",
)
ejes[1].set(
    title="Mejor modelo no lineal", xlabel="cnt real", ylabel="cnt predicho"
)
ejes[1].legend()
ejes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Conclusiones

- La evaluación se hace sobre datos que no participaron en el ajuste ni en la normalización.
- La expansión cuadrática puede representar curvaturas que la regresión lineal no captura, aunque no modela todas las interacciones posibles entre hora, clima y calendario.
- `torch.linalg.lstsq` proporciona una referencia determinista para comprobar el resultado de `nn.Linear` entrenado con Adam.
